<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

## 컴퓨터과학 예제: 작업 배정

### 1 · 문제와 정식화(problem formulation)

- 네 작업을 두 컴퓨터에 나누어 모든 작업이 빨리 끝나게 한다.
- 두 컴퓨터는 속도가 같고 동시에 시작한다.
- 각 컴퓨터는 맡은 작업을 하나씩 처리하며, 작업 분할(task splitting)과 통신 시간(communication time)은 고려하지 않는다.

| 작업 | A | B | C | D |
|:---|:---|:---|:---|:---|
| 처리 시간(processing time) (초) | 8 | 7 | 6 | 5 |

- 의사결정벡터(decision vector)는 $\boldsymbol{x}=(x_1,x_2,x_3,x_4)$이며, 각 성분(component)은 A·B·C·D의 담당 컴퓨터 번호이다.
- $x_i=0$이면 컴퓨터 0에, $x_i=1$이면 컴퓨터 1에 맡긴다.
- 두 컴퓨터의 부하(load) $L_0$와 $L_1$은 각각 맡은 작업 시간의 합이다.

$$
L_0=8(1-x_1)+7(1-x_2)+6(1-x_3)+5(1-x_4)
$$

$$
L_1=8x_1+7x_2+6x_3+5x_4
$$

- 더 오래 일하는 컴퓨터의 시간을 최소화(minimization)한다.

$$
\min_{\boldsymbol{x}}\max(L_0,L_1)
\qquad\text{subject to}\quad x_i\in\{0,1\}\;(i=1,2,3,4)
$$

- 그림의 배정은 15초와 11초로 나뉘므로 전체 완료 시간(makespan)은 15초이다.

<div class="course-figure" style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;">
<img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/3c17b9cda768a9d4f9aaca66bc0b7d01e882a5da/05-2_metaheuristic_optimization_genetic_algorithm/assets/task_assignment.svg" alt="Two processors start together. Loads are 15 and 11 seconds, so completion takes 15 seconds." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

### 2 · 코드 실행

- 공통 알고리듬(algorithm)을 정의한 뒤, 아래 셀에서 데이터(data)와 평가 함수(evaluation function)를 넣어 실행한다.


In [ ]:
import random

def genetic_algorithm(initial_population, gene_values, score, generations=20, seed=0):
    selection_top_k = 2
    crossover_rate = 1.0
    mutation_rate = 0.3
    elite_count = 1
    blend_weight = 0.5
    mutation_scale = 0.1
    bounds = (-2.0, 2.0)

    # gene_values: allowed discrete values; None: real-valued genes.
    rng = random.Random(seed)
    population = initial_population.copy()
    population_size = len(population)

    for generation in range(generations):
        # Rank all candidates from highest to lowest score.
        ranked = []
        remaining = population.copy()
        while len(remaining) > 0:
            best = remaining[0]
            for candidate in remaining:
                if score(candidate) > score(best):
                    best = candidate
            ranked.append(best)
            remaining.remove(best)

        # Elitism: copy the best candidates without changing them.
        next_population = []
        for i in range(elite_count):
            next_population.append(ranked[i].copy())

        # Selection: choose each parent randomly from the top candidates.
        parents = ranked[:selection_top_k]
        while len(next_population) < population_size:
            parent_a = rng.choice(parents)
            parent_b = rng.choice(parents)
            child = parent_a.copy()

            # Crossover: weighted average for real genes; join parts otherwise.
            if rng.random() < crossover_rate:
                if gene_values is None:
                    for i in range(len(child)):
                        child[i] = (blend_weight * parent_a[i]
                                    + (1 - blend_weight) * parent_b[i])
                else:
                    cut = rng.randrange(1, len(child))
                    for i in range(cut, len(child)):
                        child[i] = parent_b[i]

            # Mutation: change one randomly selected gene.
            if rng.random() < mutation_rate:
                i = rng.randrange(len(child))
                if gene_values is None:
                    child[i] += rng.uniform(-mutation_scale, mutation_scale)
                    if child[i] < bounds[0]:
                        child[i] = bounds[0]
                    if child[i] > bounds[1]:
                        child[i] = bounds[1]
                else:
                    alternatives = []
                    for value in gene_values:
                        if value != child[i]:
                            alternatives.append(value)
                    child[i] = rng.choice(alternatives)

            # Additional constraints: replace an infeasible child with its parent.
            if score(child) == float("-inf"):
                child = parent_a.copy()
            next_population.append(child)

        population = next_population

    best = population[0]
    for candidate in population:
        if score(candidate) > score(best):
            best = candidate
    return best.copy()


In [ ]:
durations = [8, 7, 6, 5]

def evaluate(decision):
    loads = [0, 0]
    for i in range(4):
        loads[decision[i]] += durations[i]
    return {"loads": loads, "completion_time": max(loads)}

def score(decision):
    return -evaluate(decision)["completion_time"]

initial_population = [[0, 0, 0, 0], [1, 1, 1, 1],
                      [0, 0, 1, 1], [1, 1, 0, 0]]
best = genetic_algorithm(initial_population, [0, 1], score, seed=2)
result = evaluate(best)
print("Decision:", best)
print("Loads (s):", result["loads"])
print("Completion time (s):", result["completion_time"])
print("Stop reason: generation limit")


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

- 결과는 A·B와 C·D를 나누는 배정이다.
- 두 컴퓨터의 부하는 15초와 11초이므로 전체 완료 시간은 15초이다.
- A·D와 B·C로 나누면 총작업량(total workload) 26초의 절반인 하한(lower bound) 13초를 달성하므로, 이번 실행 결과는 최적해(optimal solution)가 아니다.
